# LTX-2.3 talking-head LoRA — three-way continuation test

This notebook runs the `avatar-prototype` branch on an A100 80 GB and compares three conditioning policies under the same external audio, portrait, seed, and community talking-head LoRA: independent original-portrait reset, one-frame last-frame I2V chaining, and exact four-latent continuation with an identity anchor.

Run cells in order. The notebook transcribes the input once, builds audio-window-aligned LoRA prompts, runs all three variants, concatenates non-latent chunks for review, and displays comparable diagnostics. The LoRA is character-specific and 512×512 is outside its recommended 1280×704 image+audio training distribution, so this is a conditioning-strategy experiment rather than proof of arbitrary-portrait support.

In [ ]:
# @title 1. Inspect A100 runtime and set process environment
import os
import subprocess

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

subprocess.run(["nvidia-smi"], check=True)
subprocess.run(["free", "-h"], check=True)
subprocess.run(["df", "-h", "/content"], check=True)

In [ ]:
# @title 2. Clone or update the avatar-prototype branch
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Yuvrajxms09/LTX-2.git"
BRANCH = "avatar-prototype"
REPO = Path("/content/LTX-2")

if (REPO / ".git").exists():
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO)], check=True)

head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"], text=True).strip()
runner = REPO / "packages/ltx-pipelines/src/ltx_pipelines/avatar/runner.py"
assert runner.exists(), runner
print("checked out:", head)
print("three-way talking-head experiment loaded from current branch head")
os.chdir(REPO)
print("working directory:", Path.cwd())

In [ ]:
%%bash
# @title 3. Install the complete locked workspace environment
set -euxo pipefail
python -m pip install -q --upgrade pip uv

uv export \
  --directory /content/LTX-2 \
  --frozen \
  --all-packages \
  --all-groups \
  --no-hashes \
  --no-emit-workspace \
  --no-emit-package ltx-kernels \
  --output-file /content/LTX-2/requirements-colab-all.txt

uv pip install --system --requirement /content/LTX-2/requirements-colab-all.txt
uv pip install --system --no-deps \
  -e /content/LTX-2/packages/ltx-core \
  -e /content/LTX-2/packages/ltx-pipelines \
  -e /content/LTX-2/packages/ltx-trainer

export TORCH_CUDA_ARCH_LIST=8.0
uv pip install --system -e /content/LTX-2/packages/ltx-kernels --no-deps --no-build-isolation

python - <<'PY'
import torch
import ltx_core
import ltx_kernels
import ltx_pipelines.avatar.runner
import ltx_trainer

print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("gpu:", torch.cuda.get_device_name())
print("vram_gib:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
print("all LTX workspace packages imported")
PY

In [ ]:
# @title 4. Authenticate and download model, text encoder, and talking-head LoRA
from google.colab import userdata
from huggingface_hub import hf_hub_download, login, snapshot_download
from pathlib import Path

login(token=userdata.get("HF_TOKEN"), add_to_git_credential=False)

model_dir = Path("/content/LTX-2/models/ltx-2.3")
gemma_dir = Path("/content/LTX-2/models/gemma-3-12b")
lora_dir = Path("/content/LTX-2/models/loras")
model_dir.mkdir(parents=True, exist_ok=True)
lora_dir.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = hf_hub_download(
    repo_id="Lightricks/LTX-2.3",
    filename="ltx-2.3-22b-distilled-1.1.safetensors",
    local_dir=model_dir,
)
GEMMA_ROOT = snapshot_download(
    repo_id="google/gemma-3-12b-it-qat-q4_0-unquantized",
    local_dir=gemma_dir,
)
TALKING_HEAD_LORA_PATH = hf_hub_download(
    repo_id="elix3r/LTX-2.3-22b-AV-LoRA-talking-head",
    filename="LTX-2.3-22b-AV-LoRA-talking-head-v1.safetensors",
    local_dir=lora_dir,
)

print("checkpoint:", CHECKPOINT_PATH)
print("gemma:", GEMMA_ROOT)
print("talking-head LoRA:", TALKING_HEAD_LORA_PATH)

In [ ]:
# @title 5. Select inputs; upload only when the configured path is absent
from google.colab import files
from pathlib import Path

IMAGE_PATH = "/content/LTX-2/rdj_close_up (1).jpg" # @param {type:"string"}
AUDIO_PATH = "/content/LTX-2/14s_love2.mp3" # @param {type:"string"}

def upload_one(label, destination_dir):
    print(f"Upload {label}")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError(f"Expected exactly one {label} file")
    name, data = next(iter(uploaded.items()))
    destination = Path(destination_dir) / Path(name).name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(data)
    return str(destination)

if not Path(IMAGE_PATH).is_file():
    IMAGE_PATH = upload_one("reference image", "/content/LTX-2/inputs")
if not Path(AUDIO_PATH).is_file():
    AUDIO_PATH = upload_one("driving audio", "/content/LTX-2/inputs")

print("image:", IMAGE_PATH)
print("audio:", AUDIO_PATH)

In [ ]:
# @title 6. Transcribe the audio and write the shared LoRA configuration
import gc
import json
import subprocess
from pathlib import Path

def first_existing(*values):
    for value in values:
        if value and Path(str(value)).exists():
            return str(Path(str(value)))
    return str(values[-1])

CHECKPOINT_PATH = first_existing(
    globals().get("CHECKPOINT_PATH"),
    "/content/LTX-2/models/ltx-2.3/ltx-2.3-22b-distilled-1.1.safetensors",
)
GEMMA_ROOT = first_existing(
    globals().get("GEMMA_ROOT"),
    "/content/LTX-2/models/gemma-3-12b",
)
TALKING_HEAD_LORA_PATH = first_existing(
    globals().get("TALKING_HEAD_LORA_PATH"),
    "/content/LTX-2/models/loras/LTX-2.3-22b-AV-LoRA-talking-head-v1.safetensors",
)
IMAGE_PATH = first_existing(
    globals().get("IMAGE_PATH"),
    "/content/LTX-2/rdj_close_up (1).jpg",
    "/content/LTX-2/inputs/avatar.png",
)
AUDIO_PATH = first_existing(
    globals().get("AUDIO_PATH"),
    "/content/LTX-2/14s_love2.mp3",
    "/content/LTX-2/inputs/tts.wav",
)

required_paths = {
    "checkpoint": Path(CHECKPOINT_PATH),
    "gemma": Path(GEMMA_ROOT),
    "talking_head_lora": Path(TALKING_HEAD_LORA_PATH),
    "image": Path(IMAGE_PATH),
    "audio": Path(AUDIO_PATH),
}
missing = {name: str(path) for name, path in required_paths.items() if not path.exists()}
if missing:
    raise FileNotFoundError(f"Missing required inputs: {missing}. Run Cells 4 and 5 first.")

AUDIO_DURATION = float(subprocess.check_output([
    "ffprobe", "-v", "error", "-show_entries", "format=duration",
    "-of", "default=noprint_wrappers=1:nokey=1", str(AUDIO_PATH),
], text=True).strip())
MANUAL_TRANSCRIPT = "" # @param {type:"string"}
ASR_MODEL = "openai/whisper-large-v3-turbo" # @param {type:"string"}

if MANUAL_TRANSCRIPT.strip():
    TRANSCRIPT_SEGMENTS = [{"start": 0.0, "end": AUDIO_DURATION, "text": MANUAL_TRANSCRIPT.strip()}]
else:
    import torch
    from transformers import pipeline

    asr = pipeline(
        "automatic-speech-recognition",
        model=ASR_MODEL,
        torch_dtype=torch.float16,
        device=0,
    )
    transcription = asr(str(AUDIO_PATH), return_timestamps="word")
    TRANSCRIPT_SEGMENTS = []
    for item in transcription.get("chunks", []):
        start, end = item.get("timestamp", (None, None))
        if start is None:
            continue
        TRANSCRIPT_SEGMENTS.append({
            "start": float(start),
            "end": AUDIO_DURATION if end is None else float(end),
            "text": item.get("text", "").strip(),
        })
    if not TRANSCRIPT_SEGMENTS:
        TRANSCRIPT_SEGMENTS = [{
            "start": 0.0,
            "end": AUDIO_DURATION,
            "text": transcription["text"].strip(),
        }]
    del asr
    gc.collect()
    torch.cuda.empty_cache()

print("audio duration:", AUDIO_DURATION)
print(json.dumps(TRANSCRIPT_SEGMENTS, indent=2))

CONFIG_PATH = Path("/content/LTX-2/avatar-talking-head.toml")
BASE_PROMPT = "OHWXPERSON, a stable front-facing portrait of a person facing the camera. The camera, focal length, face position, lighting, clothing, and background remain unchanged. The person talks naturally with subtle head motion."

CONFIG_PATH.write_text(f'''[model]
checkpoint_path = {json.dumps(str(CHECKPOINT_PATH))}
gemma_root = {json.dumps(str(GEMMA_ROOT))}
offload = "cpu"
warm_transformer = true
compile = false

[[model.loras]]
path = {json.dumps(str(TALKING_HEAD_LORA_PATH))}
strength = 1.0

[input]
image_path = {json.dumps(str(IMAGE_PATH))}
audio_path = {json.dumps(str(AUDIO_PATH))}
prompt = {json.dumps(BASE_PROMPT)}
enhance_prompt = false

[generation]
width = 512
height = 512
frame_rate = 25.0
generation_frames = 121
overlap_frames = 25
continuation_mode = "latent-prefix"
reference_strength = 1.0
overlap_strength = 1.0
identity_anchor_strength = 0.5
seed = 10
seed_stride = 1
sigmas = [1.0, 0.99375, 0.9875, 0.98125, 0.975, 0.909375, 0.725, 0.421875, 0.0]
max_chunks = 4

[output]
directory = "/content/LTX-2/outputs/avatar-talking-head-placeholder"
crf = 19
preset = "veryfast"
save_conditioning_frames = true
allow_existing = false

[diagnostics]
log_level = "INFO"
jsonl_metrics = true
synchronize_cuda = true
log_denoising_steps = true
tensor_statistics = false
''', encoding="utf-8")

print(CONFIG_PATH.read_text())

In [ ]:
# @title 7. Define transcript-aligned three-way experiment runner
import math
import subprocess
import sys
import time
from pathlib import Path

def planned_windows(continuation_mode, overlap_frames, max_chunks):
    raw_frames = AUDIO_DURATION * 25.0
    if continuation_mode == "latent-prefix":
        total_frames = max(1, math.floor((raw_frames - 1) / 8 + 0.5) * 8 + 1)
    else:
        total_frames = max(1, math.ceil(raw_frames))
    windows = []
    emitted = 0
    while emitted < total_frames and len(windows) < max_chunks:
        overlap = 0 if not windows else min(overlap_frames, emitted)
        start_frame = emitted - overlap
        windows.append((start_frame / 25.0, (start_frame + 121) / 25.0))
        emitted += min(121 - overlap, total_frames - emitted)
    return windows

def transcript_for_window(start, end):
    text = " ".join(
        segment["text"]
        for segment in TRANSCRIPT_SEGMENTS
        if segment["text"] and segment["start"] < end and segment["end"] > start
    ).strip()
    if not text:
        raise ValueError(f"No transcript text overlaps audio window {start:.2f}-{end:.2f}s")
    return text

def chunk_prompts(continuation_mode, overlap_frames, max_chunks):
    return [
        f'{BASE_PROMPT} The person is talking, and he says: "{transcript_for_window(start, end)}"'
        for start, end in planned_windows(continuation_mode, overlap_frames, max_chunks)
    ]

def ensure_combined(run_dir):
    run_dir = Path(run_dir)
    combined = run_dir / "combined.mp4"
    if combined.exists():
        return combined
    chunks = sorted(run_dir.glob("chunk_*.mp4"))
    if not chunks:
        raise FileNotFoundError(f"No chunk MP4s found in {run_dir}")
    concat_list = run_dir / "concat.txt"
    concat_list.write_text("".join(f"file '{path.resolve()}'\n" for path in chunks))
    subprocess.run([
        "ffmpeg", "-y", "-v", "error", "-f", "concat", "-safe", "0",
        "-i", str(concat_list), "-c", "copy", str(combined),
    ], check=True)
    return combined

def run_experiment(
    label,
    *,
    continuation_mode,
    overlap_frames,
    overlap_strength,
    identity_anchor_strength,
    max_chunks,
):
    stamp = time.strftime("%Y%m%d-%H%M%S")
    output_dir = Path(f"/content/LTX-2/outputs/{label}-{stamp}")
    prompts = chunk_prompts(continuation_mode, overlap_frames, max_chunks)
    cmd = [
        sys.executable, "-m", "ltx_pipelines.avatar.runner",
        "--config", str(CONFIG_PATH),
        "--set", f'generation.continuation_mode={json.dumps(continuation_mode)}',
        "--set", "generation.generation_frames=121",
        "--set", f"generation.overlap_frames={overlap_frames}",
        "--set", f"generation.overlap_strength={overlap_strength}",
        "--set", f"generation.identity_anchor_strength={identity_anchor_strength}",
        "--set", f"generation.max_chunks={max_chunks}",
        "--set", f"input.chunk_prompts={json.dumps(prompts)}",
        "--set", f'output.directory="{output_dir}"',
    ]
    print(json.dumps({"label": label, "windows": planned_windows(continuation_mode, overlap_frames, max_chunks), "prompts": prompts}, indent=2))
    subprocess.run(cmd, cwd="/content/LTX-2", check=True)
    ensure_combined(output_dir)
    return output_dir

subprocess.run([
    sys.executable, "-m", "ltx_pipelines.avatar.runner",
    "--config", str(CONFIG_PATH), "--dry-run",
], cwd="/content/LTX-2", check=True)

In [ ]:
# @title 8. Run all three continuation strategies
RUN_THREE_WAY_COMPARISON = True # @param {type:"boolean"}
MAX_CHUNKS = 4 # @param {type:"integer"}

RUNS = {}
if RUN_THREE_WAY_COMPARISON:
    variants = [
        {
            "label": "talking-lora-reference-reset",
            "continuation_mode": "reference-reset",
            "overlap_frames": 0,
            "overlap_strength": 1.0,
            "identity_anchor_strength": 0.0,
        },
        {
            "label": "talking-lora-last-frame-i2v",
            "continuation_mode": "image-keyframes",
            "overlap_frames": 1,
            "overlap_strength": 1.0,
            "identity_anchor_strength": 0.0,
        },
        {
            "label": "talking-lora-4lat-exact-anchor05",
            "continuation_mode": "latent-prefix",
            "overlap_frames": 25,
            "overlap_strength": 1.0,
            "identity_anchor_strength": 0.5,
        },
    ]
    for variant in variants:
        label = variant.pop("label")
        RUNS[label] = run_experiment(label, max_chunks=MAX_CHUNKS, **variant)
    print(json.dumps({key: str(value) for key, value in RUNS.items()}, indent=2))

In [ ]:
# @title 9. Display all three combined videos and chunk diagnostics
import json
from IPython.display import Video, display

def review_run(run_dir):
    run_dir = Path(run_dir)
    manifest = json.loads((run_dir / "manifest.json").read_text())
    summary = {
        "status": manifest.get("status"),
        "audio_duration_seconds": manifest.get("audio_duration_seconds"),
        "wall_seconds": manifest.get("wall_seconds"),
        "time_to_first_chunk_seconds": manifest.get("time_to_first_chunk_seconds"),
        "combined_output": manifest.get("combined_output"),
    }
    print(json.dumps(summary, indent=2))
    for chunk in manifest.get("completed_chunks", []):
        print(json.dumps({
            "chunk": chunk["index"],
            "emitted_frames": chunk["emitted_frames"],
            "identity_anchor_enabled": chunk.get("identity_anchor_enabled"),
            "latent_fusion": chunk.get("latent_fusion"),
            "continuity": chunk.get("continuity"),
        }, indent=2))
    combined = ensure_combined(run_dir)
    display(Video(str(combined), embed=True, width=512))
    return manifest

MANIFESTS = {}
for label, run_dir in RUNS.items():
    print("\n===", label, "===")
    MANIFESTS[label] = review_run(run_dir)

In [ ]:
# @title 10. Build a compact comparison table
rows = []
for label, manifest in MANIFESTS.items():
    chunks = manifest.get("completed_chunks", [])
    boundaries = [
        chunk.get("continuity", {}).get("boundary_rmse")
        for chunk in chunks[1:]
        if chunk.get("continuity", {}).get("boundary_rmse") is not None
    ]
    rows.append({
        "variant": label,
        "mode": manifest["config"]["generation"]["continuation_mode"],
        "chunks": len(chunks),
        "emitted_frames": sum(chunk["emitted_frames"] for chunk in chunks),
        "wall_seconds": manifest.get("wall_seconds"),
        "mean_boundary_rmse": sum(boundaries) / len(boundaries) if boundaries else None,
        "last_prefix_max_abs": chunks[-1].get("continuity", {}).get("latent_prefix_max_abs") if chunks else None,
    })

try:
    import pandas as pd
    display(pd.DataFrame(rows))
except ImportError:
    print(json.dumps(rows, indent=2))

In [ ]:
# @title 11. Print output paths for download
for label, run_dir in RUNS.items():
    print(label)
    print("  combined:", ensure_combined(run_dir))
    print("  manifest:", Path(run_dir) / "manifest.json")
    print("  metrics:", Path(run_dir) / "metrics.jsonl")

## Interpretation

- `reference-reset` answers whether repeatedly conditioning on the original portrait prevents cumulative zoom and face drift. Its joins can still jump because each clip is independent.
- `image-keyframes` answers whether a single previous last frame is sufficient. It can accumulate scale/composition errors because each output becomes the next input.
- `latent-prefix` answers whether exact model-native temporal state gives the best motion continuity. Its negative-index anchor is the only additional identity reference in this variant.
- Compare mouth activity and lip timing inside chunks separately from camera/identity drift across boundaries.
- The selected LoRA is character-specific. If it changes the RDJ portrait toward its trained subject, that is a LoRA limitation, not a continuation bug.
- The non-latent `combined.mp4` files are stream-copy concatenations of independently decoded chunks. The latent-prefix `combined.mp4` is a single decode of the fused latent timeline.